<a href="https://colab.research.google.com/github/Cyberpunk-San/ML-practice/blob/Decision-Tree/Decision_Tree_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Decision Tree- Scratch

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Data processing

In [ ]:
import numpy as np
from collections import Counter

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class DecisionTreeScratch:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # 1. Stopping Criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # 2. Find Best Split
        feat_idxs = np.arange(n_features)
        best_feat, best_thresh = self._best_criteria(X, y, feat_idxs)

        # 3. Recursive Expansion
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)

    def _best_criteria(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold
        return split_idx, split_thresh

    def _information_gain(self, y, X_column, split_thresh):
        # Parent Entropy
        parent_entropy = self._entropy(y)
        # Create Children
        left_idxs, right_idxs = self._split(X_column, split_thresh)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        # Weighted Average Child Entropy
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_entropy - child_entropy

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

Decision Tree Class

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

clf = DecisionTreeScratch(max_depth=10)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = np.sum(y_pred == y_test) / len(y_test)
print(f"Decision Tree Accuracy: {acc * 100:.2f}%")

Decision Tree Accuracy: 100.00%


Implementation

In [ ]:
import plotly.graph_objects as go

# Let's map the predictions over a 3D grid
x_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 20)
y_range = np.linspace(X[:, 1].min(), X[:, 1].max(), 20)
z_range = np.linspace(X[:, 2].min(), X[:, 2].max(), 20)
xx, yy, zz = np.meshgrid(x_range, y_range, z_range)

grid = np.c_[xx.ravel(), yy.ravel(), zz.ravel(), np.zeros(len(xx.ravel()))] # Pad with 0 for 4th feature
preds = clf.predict(grid).reshape(xx.shape)

fig = go.Figure(data=go.Volume(
    x=xx.ravel(), y=yy.ravel(), z=zz.ravel(),
    value=preds.ravel(),
    isomin=0, isomax=2,
    opacity=0.1,
    surface_count=3,
    colorscale='Viridis'
))
fig.update_layout(title="Decision Tree 3D: Categorical 'Boxes' in Feature Space")
fig.show()

Visualization

*By completing the _grow_tree recursion and adding the Information Gain logic, we have built a model that partitions data using Entropy. Unlike our previous geometric models (SVM/KNN) that care about distance, the Decision Tree cares about purity. It exhaustively searches through every feature and every possible threshold to find the "cut" that most effectively separates the classes. The 3D visualization reveals the "blocky" nature of these decisions—since each split is axis-aligned, the model effectively carves the feature space into hyper-rectangles. This makes the model incredibly fast to execute and easy to explain, though it remains prone to overfitting without depth limits.*